In [1]:
# word2vec
# 推論ベースの手法とニューラルネットワーク
# カウントベースの手法の問題点
# 推論ベースの手法の概要
# ニューラルネットワークにおける単語の処理方法

In [3]:
# 全結合層による返還
import numpy as np

rng = np.random.default_rng()

c = np.array([[1, 0, 0, 0, 0, 0, 0]]) # 入力
W = rng.random((7, 3))                # 重み
h = c @ W                             # 中間ノード
print(h)

[[0.83956975 0.65320718 0.44902003]]


In [5]:
# MatMulレイヤの実装
import sys
sys.path.append("..")
from common.layers import MatMul

c = np.array([[1, 0, 0, 0, 0, 0, 0]])
W = rng.random((7, 3))
layer = MatMul(W)
h = layer.forward(c)
print(h)

[[0.82959655 0.69097331 0.9665133 ]]


In [6]:
# シンプルなword2vec
# CBOWモデルの推論処理
import sys
sys.path.append("..")
from common.layers import MatMul

# サンプルのコンテキストデータ
c0 = np.array([[1, 0, 0, 0, 0, 0, 0]])
c1 = np.array([[0, 0, 1, 0, 0, 0, 0]])

# 重みの初期化
W_in = rng.random((7, 3))
W_out = rng.random((3, 7))

# レイヤの生成
in_layer0 = MatMul(W_in)
in_layer1 = MatMul(W_in)
out_layer = MatMul(W_out)

# 順伝播
h0 = in_layer0.forward(c0)
h1 = in_layer1.forward(c1)
h = 0.5 * (h0 + h1)
s = out_layer.forward(h)

print(s)

[[0.82040484 0.93395256 0.93614891 0.74501856 0.77425556 0.94557125
  0.51948465]]


In [7]:
# CBOWモデルの学習
# word2vecの重みと分散表現
# コンテキストとターゲット

In [9]:
# コーパスのテキストを単語IDに変換する
import sys
sys.path.append("..")
from common.util import preprocess

text = "You say goodbye and I say hello."
corpus, word_to_id, id_to_word = preprocess(text)
print(corpus)
print()
print(id_to_word)

[0 1 2 3 4 1 5 6]

{0: 'you', 1: 'say', 2: 'goodbye', 3: 'and', 4: 'i', 5: 'hello', 6: '.'}


In [12]:
# コンテキストとターゲットを作成する関数を実装
def create_contexts_target(corpus, window_size=1):
    target = corpus[window_size:-window_size]
    contexts = []

    for idx in range(window_size, len(corpus)-window_size):
        cs = []
        for t in range(-window_size, window_size + 1):
            if t == 0:
                continue
            cs.append(corpus[idx + t])
        contexts.append(cs)

    return np.array(contexts), np.array(target)

contexts, target = create_contexts_target(corpus, window_size=1)
print(contexts)
print()
print(target)

[[0 2]
 [1 3]
 [2 4]
 [3 1]
 [4 5]
 [1 6]]

[1 2 3 4 1 5]


In [14]:
# one-hot表現への変換
import sys
sys.path.append("..")
from common.util import preprocess, create_contexts_target, convert_one_hot

text = "You say goodbye and I say hello."
corpus, word_to_id, id_to_word = preprocess(text)

contexts, target = create_contexts_target(corpus, window_size=1)

vocab_size = len(word_to_id)
target = convert_one_hot(target, vocab_size)
contexts = convert_one_hot(contexts, vocab_size)

In [15]:
print(contexts.shape)
print(target.shape)

(6, 2, 7)
(6, 7)
